### 4. Download dataset

You've provided a link to the 'RSNA Bone Age' dataset. Let's download this dataset.

In [1]:
import kagglehub

# Download the 'RSNA Bone Age' dataset
dataset_path = kagglehub.dataset_download('kmader/rsna-bone-age')

print("Path to RSNA Bone Age dataset files:", dataset_path)

Path to RSNA Bone Age dataset files: /kaggle/input/datasets/kmader/rsna-bone-age


Let's list the contents of the downloaded RSNA Bone Age dataset to understand its structure.

In [2]:
#import os

#print("Contents of the RSNA Bone Age dataset:")
#for dirname, _, filenames in os.walk(dataset_path):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

Let's load the `boneage-training-dataset.csv` into a pandas DataFrame and examine its structure.

In [3]:
import pandas as pd
import os

training_csv_path = os.path.join(dataset_path, 'boneage-training-dataset.csv')
train_df = pd.read_csv(training_csv_path)

print("Training DataFrame Info:")
train_df.info()

print("\nFirst 5 rows of Training DataFrame:")
display(train_df.head())

Training DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12611 entries, 0 to 12610
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       12611 non-null  int64
 1   boneage  12611 non-null  int64
 2   male     12611 non-null  bool 
dtypes: bool(1), int64(2)
memory usage: 209.5 KB

First 5 rows of Training DataFrame:


,id,boneage,male
0,1377,180,False
1,1378,12,False
2,1379,94,False
3,1380,120,True
4,1381,82,False


Now, let's do the same for the `boneage-test-dataset.csv`.

In [4]:
test_csv_path = os.path.join(dataset_path, 'boneage-test-dataset.csv')
test_df = pd.read_csv(test_csv_path)

print("Test DataFrame Info:")
test_df.info()

print("\nFirst 5 rows of Test DataFrame:")
display(test_df.head())

Test DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Case ID  200 non-null    int64 
 1   Sex      200 non-null    object
dtypes: int64(1), object(1)
memory usage: 3.3+ KB

First 5 rows of Test DataFrame:


,Case ID,Sex
0,4360,M
1,4361,M
2,4362,M
3,4363,M
4,4364,M


### 6. Data Preprocessing
As noted earlier, there are inconsistencies in column names and types between the training and test DataFrames. Let's address these.

In [5]:
import numpy as np

# Harmonize column names and types

# 1. For test_df, rename 'Case ID' to 'id' and 'Sex' to 'male'
test_df = test_df.rename(columns={'Case ID': 'id', 'Sex': 'male'})

# 2. Convert 'male' column in test_df to boolean (True for 'M', False for 'F')
test_df['male'] = test_df['male'].apply(lambda x: True if x == 'M' else False)

# Display updated DataFrames to verify changes
print("Updated Training DataFrame Info:")
train_df.info()
print("\nUpdated Test DataFrame Info:")
test_df.info()

print("\nFirst 5 rows of Updated Training DataFrame:")
display(train_df.head())
print("\nFirst 5 rows of Updated Test DataFrame:")
display(test_df.head())

Updated Training DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12611 entries, 0 to 12610
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       12611 non-null  int64
 1   boneage  12611 non-null  int64
 2   male     12611 non-null  bool 
dtypes: bool(1), int64(2)
memory usage: 209.5 KB

Updated Test DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      200 non-null    int64
 1   male    200 non-null    bool 
dtypes: bool(1), int64(1)
memory usage: 1.9 KB

First 5 rows of Updated Training DataFrame:


,id,boneage,male
0,1377,180,False
1,1378,12,False
2,1379,94,False
3,1380,120,True
4,1381,82,False



First 5 rows of Updated Test DataFrame:


,id,male
0,4360,True
1,4361,True
2,4362,True
3,4363,True
4,4364,True


Now that our tabular data is harmonized, let's add the full image paths to both DataFrames. This will allow us to easily load the corresponding X-ray images.

In [6]:
# Create full image paths for training and test data
train_image_dir = os.path.join(dataset_path, 'boneage-training-dataset', 'boneage-training-dataset')
test_image_dir = os.path.join(dataset_path, 'boneage-test-dataset', 'boneage-test-dataset')

train_df['path'] = train_df['id'].apply(lambda x: os.path.join(train_image_dir, f'{x}.png'))
test_df['path'] = test_df['id'].apply(lambda x: os.path.join(test_image_dir, f'{x}.png'))

# Verify that the paths exist and display some examples
print("Example training image path:", train_df['path'].iloc[0])
print("Example test image path:", test_df['path'].iloc[0])

print("\nFirst 5 rows of Training DataFrame with paths:")
display(train_df.head())
print("\nFirst 5 rows of Test DataFrame with paths:")
display(test_df.head())

Example training image path: /kaggle/input/datasets/kmader/rsna-bone-age/boneage-training-dataset/boneage-training-dataset/1377.png
Example test image path: /kaggle/input/datasets/kmader/rsna-bone-age/boneage-test-dataset/boneage-test-dataset/4360.png

First 5 rows of Training DataFrame with paths:


,id,boneage,male,path
0,1377,180,False,/kaggle/input/datasets/kmader/rsna-bone-age/bo...
1,1378,12,False,/kaggle/input/datasets/kmader/rsna-bone-age/bo...
2,1379,94,False,/kaggle/input/datasets/kmader/rsna-bone-age/bo...
3,1380,120,True,/kaggle/input/datasets/kmader/rsna-bone-age/bo...
4,1381,82,False,/kaggle/input/datasets/kmader/rsna-bone-age/bo...



First 5 rows of Test DataFrame with paths:


,id,male,path
0,4360,True,/kaggle/input/datasets/kmader/rsna-bone-age/bo...
1,4361,True,/kaggle/input/datasets/kmader/rsna-bone-age/bo...
2,4362,True,/kaggle/input/datasets/kmader/rsna-bone-age/bo...
3,4363,True,/kaggle/input/datasets/kmader/rsna-bone-age/bo...
4,4364,True,/kaggle/input/datasets/kmader/rsna-bone-age/bo...


### 7. Image Preprocessing

Now that we have the paths to our images, let's load and preprocess them. This will involve resizing, normalizing pixel values, and preparing them for input into our deep learning model.

In [7]:
"""
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, concatenate, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam

def build_improved_boneage_model(input_shape=(IMG_WIDTH, IMG_HEIGHT, 3), gender_input_shape=(1,)):
    # 1. Pre-trained Base Model (EfficientNetB3)
    base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=input_shape)

    # FREEZE the base model layers to prevent destroying pre-trained weights early on
    for layer in base_model.layers:
        layer.trainable = False

    # Image input branch
    image_input = Input(shape=input_shape, name='image_input')
    x = base_model(image_input)
    x = GlobalAveragePooling2D()(x) # Better than Flatten for deep networks
    x = BatchNormalization()(x)     # Stabilize training

    # Gender input branch
    gender_input = Input(shape=gender_input_shape, name='gender_input')

    # Concatenate features
    combined_input = concatenate([x, gender_input])

    # Fully connected layers
    z = Dense(256, activation='relu')(combined_input)
    z = BatchNormalization()(z)     # Stabilize training
    z = Dropout(0.4)(z)
    z = Dense(128, activation='relu')(z)
    z = BatchNormalization()(z)
    z = Dropout(0.3)(z)

    # Output layer for regression
    output = Dense(1, activation='linear', name='boneage_output')(z)

    # Create the model
    improved_model = Model(inputs=[image_input, gender_input], outputs=output)

    # Compile the model - using a slightly higher learning rate since we are only training the top layers
    improved_model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_absolute_error', metrics=['mean_absolute_error'])

    return improved_model

# Build and view the improved model
improved_model = build_improved_boneage_model()
improved_model.summary()
"""

"\nfrom tensorflow.keras.applications import EfficientNetB3\nfrom tensorflow.keras.models import Model\nfrom tensorflow.keras.layers import Input, Dense, Dropout, concatenate, GlobalAveragePooling2D, BatchNormalization\nfrom tensorflow.keras.optimizers import Adam\n\ndef build_improved_boneage_model(input_shape=(IMG_WIDTH, IMG_HEIGHT, 3), gender_input_shape=(1,)):\n    # 1. Pre-trained Base Model (EfficientNetB3)\n    base_model = EfficientNetB3(weights='imagenet', include_top=False, input_shape=input_shape)\n\n    # FREEZE the base model layers to prevent destroying pre-trained weights early on\n    for layer in base_model.layers:\n        layer.trainable = False\n\n    # Image input branch\n    image_input = Input(shape=input_shape, name='image_input')\n    x = base_model(image_input)\n    x = GlobalAveragePooling2D()(x) # Better than Flatten for deep networks\n    x = BatchNormalization()(x)     # Stabilize training\n\n    # Gender input branch\n    gender_input = Input(shape=gender

In [8]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Image dimensions (Increased for better detail capture)
IMG_WIDTH = 384
IMG_HEIGHT = 384
# Reduced batch size to accommodate larger images in GPU memory
BATCH_SIZE = 16

# Create an ImageDataGenerator for training data with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values to [0, 1]
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2], # Adjust brightness for varied X-ray exposures
    horizontal_flip=False, # Removed flip to maintain left-hand orientation
    fill_mode='nearest',
    validation_split=0.2 # Use a portion of training data for validation
)

# Create an ImageDataGenerator for test data (only rescaling)
test_datagen = ImageDataGenerator(
    rescale=1./255  # Normalize pixel values to [0, 1]
)

# Flow from dataframe for training data
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='boneage',
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='raw', # For regression tasks
    subset='training',
    seed=42
)

# Flow from dataframe for validation data
validation_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col='boneage',
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    subset='validation',
    seed=42
)

# Flow from dataframe for test data
# Note: For test data, y_col is typically not available,
# so we use class_mode=None and shuffle=False
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='path',
    y_col=None, # No labels for the test set
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

print("Image preprocessing setup complete. Generators created for training, validation, and test data.")

2026-07-01 07:45:41.748387: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782891941.936828      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782891941.995183      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782891942.416430      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782891942.416477      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782891942.416481      23 computation_placer.cc:177] computation placer alr

Found 10089 validated image filenames.
Found 2522 validated image filenames.
Found 200 validated image filenames.
Image preprocessing setup complete. Generators created for training, validation, and test data.


### 12. Model Improvement: Transfer Learning with ResNet50V2

Using a model pre-trained on ImageNet allows us to leverage complex feature extraction right out of the box. We will freeze the base model initially, or train it with a very low learning rate.

In [9]:
"""
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Callbacks for better training control
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_boneage_model.keras', save_best_only=True, monitor='val_loss', mode='min', verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001, verbose=1)

# Recreate generators to output both 'boneage' and 'male' as targets
train_generator_multi = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col=['boneage', 'male'], # Request both columns
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    subset='training',
    seed=42
)

validation_generator_multi = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col=['boneage', 'male'], # Request both columns
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    subset='validation',
    seed=42
)

# Wrapper generator to split the combined targets into (gender input, boneage target)
def combined_generator(img_gen):
    for img_batch, targets_batch in img_gen:
        # Explicitly cast to float32 to fix the 'object' dtype error
        boneage_batch = targets_batch[:, 0].astype('float32')
        gender_batch = targets_batch[:, 1].astype('float32')

        # Yield a tuple of inputs ((image_batch, gender_batch), boneage_batch)
        yield ((img_batch, gender_batch), boneage_batch)

# Create combined generators for training and validation
train_data_generator = combined_generator(train_generator_multi)
validation_data_generator = combined_generator(validation_generator_multi)

EPOCHS = 50

print("Starting model training...")

history = improved_model.fit(
    train_data_generator,
    steps_per_epoch=train_generator_multi.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_data_generator,
    validation_steps=validation_generator_multi.samples // BATCH_SIZE,
    callbacks=[early_stopping, model_checkpoint, reduce_lr]
)

print("Model training complete.")
"""

'\nfrom tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau\n\n# Callbacks for better training control\nearly_stopping = EarlyStopping(monitor=\'val_loss\', patience=10, restore_best_weights=True)\nmodel_checkpoint = ModelCheckpoint(\'best_boneage_model.keras\', save_best_only=True, monitor=\'val_loss\', mode=\'min\', verbose=1)\nreduce_lr = ReduceLROnPlateau(monitor=\'val_loss\', factor=0.5, patience=5, min_lr=0.00001, verbose=1)\n\n# Recreate generators to output both \'boneage\' and \'male\' as targets\ntrain_generator_multi = train_datagen.flow_from_dataframe(\n    dataframe=train_df,\n    x_col=\'path\',\n    y_col=[\'boneage\', \'male\'], # Request both columns\n    target_size=(IMG_WIDTH, IMG_HEIGHT),\n    batch_size=BATCH_SIZE,\n    class_mode=\'raw\',\n    subset=\'training\',\n    seed=42\n)\n\nvalidation_generator_multi = train_datagen.flow_from_dataframe(\n    dataframe=train_df,\n    x_col=\'path\',\n    y_col=[\'boneage\', \'male\'], # Requ

### 9. Visualize Training History
Let's plot the training and validation loss (Mean Absolute Error) to understand how the model learned over time.

In [10]:
"""
import matplotlib.pyplot as plt

# Extract loss values from the training history
train_loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(train_loss) + 1)

# Plot the training and validation loss
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_loss, 'b-', label='Training Loss (MAE)', linewidth=2)
plt.plot(epochs, val_loss, 'r-', label='Validation Loss (MAE)', linewidth=2)

# Formatting the plot
plt.title('Model Training Evolution: Loss / MAE vs. Epochs', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Mean Absolute Error (Months)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)

# Show the plot
plt.tight_layout()
plt.show()
"""

"\nimport matplotlib.pyplot as plt\n\n# Extract loss values from the training history\ntrain_loss = history.history['loss']\nval_loss = history.history['val_loss']\nepochs = range(1, len(train_loss) + 1)\n\n# Plot the training and validation loss\nplt.figure(figsize=(10, 6))\nplt.plot(epochs, train_loss, 'b-', label='Training Loss (MAE)', linewidth=2)\nplt.plot(epochs, val_loss, 'r-', label='Validation Loss (MAE)', linewidth=2)\n\n# Formatting the plot\nplt.title('Model Training Evolution: Loss / MAE vs. Epochs', fontsize=14)\nplt.xlabel('Epoch', fontsize=12)\nplt.ylabel('Mean Absolute Error (Months)', fontsize=12)\nplt.legend(fontsize=12)\nplt.grid(True, linestyle='--', alpha=0.7)\n\n# Show the plot\nplt.tight_layout()\nplt.show()\n"

### 13. Fine-Tuning the Model

Since the model plateaued quickly, we will unfreeze the base `EfficientNetB3` layers and train the entire model end-to-end with a much lower learning rate. This allows the feature extractors to adapt specifically to the X-ray images.

In [11]:
"""
# Unfreeze all layers in the model
for layer in improved_model.layers:
    layer.trainable = True

# Recompile the model with a much lower learning rate for fine-tuning
improved_model.compile(
    optimizer=Adam(learning_rate=1e-5), # 100x smaller than the initial learning rate
    loss='mean_absolute_error', 
    metrics=['mean_absolute_error']
)

print("Model unfrozen and recompiled for fine-tuning.")
improved_model.summary()
"""

'\n# Unfreeze all layers in the model\nfor layer in improved_model.layers:\n    layer.trainable = True\n\n# Recompile the model with a much lower learning rate for fine-tuning\nimproved_model.compile(\n    optimizer=Adam(learning_rate=1e-5), # 100x smaller than the initial learning rate\n    loss=\'mean_absolute_error\', \n    metrics=[\'mean_absolute_error\']\n)\n\nprint("Model unfrozen and recompiled for fine-tuning.")\nimproved_model.summary()\n'

In [12]:
"""
# Set up new callbacks for the fine-tuning phase
fine_tune_early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
fine_tune_checkpoint = ModelCheckpoint('best_finetuned_boneage_model.keras', save_best_only=True, monitor='val_loss', mode='min', verbose=1)

FINE_TUNE_EPOCHS = 20

print("Starting fine-tuning...")

history_finetune = improved_model.fit(
    train_data_generator,
    steps_per_epoch=train_generator_multi.samples // BATCH_SIZE,
    epochs=FINE_TUNE_EPOCHS,
    validation_data=validation_data_generator,
    validation_steps=validation_generator_multi.samples // BATCH_SIZE,
    callbacks=[fine_tune_early_stopping, fine_tune_checkpoint]
)

print("Fine-tuning complete.")
"""

'\n# Set up new callbacks for the fine-tuning phase\nfine_tune_early_stopping = EarlyStopping(monitor=\'val_loss\', patience=7, restore_best_weights=True)\nfine_tune_checkpoint = ModelCheckpoint(\'best_finetuned_boneage_model.keras\', save_best_only=True, monitor=\'val_loss\', mode=\'min\', verbose=1)\n\nFINE_TUNE_EPOCHS = 20\n\nprint("Starting fine-tuning...")\n\nhistory_finetune = improved_model.fit(\n    train_data_generator,\n    steps_per_epoch=train_generator_multi.samples // BATCH_SIZE,\n    epochs=FINE_TUNE_EPOCHS,\n    validation_data=validation_data_generator,\n    validation_steps=validation_generator_multi.samples // BATCH_SIZE,\n    callbacks=[fine_tune_early_stopping, fine_tune_checkpoint]\n)\n\nprint("Fine-tuning complete.")\n'

### 10. Generate Predictions and Submit

Now we will load the best model weights, generate predictions for the test dataset (providing both the X-ray images and the 'male' boolean flag as inputs), and save the output as `submission.csv`.

### 11. Download Competition Test Data

We need to use the specific test data for the `eliva-25-medical` competition to match the expected 33 rows for submission.

In [13]:
!kaggle competitions download -c eliva-25-medical -p eliva_data
!unzip -q -o eliva_data/eliva-25-medical.zip -d eliva_data

100%|██████████████████████████████████████| 19.3M/19.3M [00:00<00:00, 44.1MB/s]



In [14]:
import os

# List the extracted files to understand the directory structure
print("Contents of eliva_data directory:")
for dirname, _, filenames in os.walk('eliva_data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Contents of eliva_data directory:
eliva_data/eliva-25-medical.zip
eliva_data/test/19.png
eliva_data/test/25.png
eliva_data/test/27.png
eliva_data/test/14.png
eliva_data/test/29.png
eliva_data/test/5.png
eliva_data/test/23.png
eliva_data/test/15.png
eliva_data/test/30.png
eliva_data/test/28.png
eliva_data/test/31.png
eliva_data/test/test.csv
eliva_data/test/32.png
eliva_data/test/7.png
eliva_data/test/13.png
eliva_data/test/17.png
eliva_data/test/33.png
eliva_data/test/20.png
eliva_data/test/26.png
eliva_data/test/1.png
eliva_data/test/10.png
eliva_data/test/16.png
eliva_data/test/11.png
eliva_data/test/8.png
eliva_data/test/2.png
eliva_data/test/22.png
eliva_data/test/24.png
eliva_data/test/9.png
eliva_data/test/6.png
eliva_data/test/4.png
eliva_data/test/21.png
eliva_data/test/12.png
eliva_data/test/18.png
eliva_data/test/3.png


In [15]:
import pandas as pd
import os
import math
import numpy as np
from tensorflow.keras.models import load_model

# Load the competition test set
comp_test_csv = 'eliva_data/test/test.csv'
comp_test_df = pd.read_csv(comp_test_csv)
print("Original Competition Test DataFrame:")
display(comp_test_df.head())

# Standardize columns to 'id' and 'male' (boolean)
if 'Case ID' in comp_test_df.columns:
    comp_test_df = comp_test_df.rename(columns={'Case ID': 'id'})
if 'Sex' in comp_test_df.columns:
    comp_test_df = comp_test_df.rename(columns={'Sex': 'male'})
    comp_test_df['male'] = comp_test_df['male'].apply(lambda x: True if x == 'M' else False)
elif 'gender' in comp_test_df.columns:
    comp_test_df['male'] = comp_test_df['gender'].apply(lambda x: True if str(x).lower() in ['m', 'male'] else False)

# Add full paths
comp_test_image_dir = 'eliva_data/test'
comp_test_df['path'] = comp_test_df['id'].apply(lambda x: os.path.join(comp_test_image_dir, str(x)))

print("\nProcessed Competition Test DataFrame:")
display(comp_test_df.head())

# Create a new generator for the 33 competition images
comp_test_generator = test_datagen.flow_from_dataframe(
    dataframe=comp_test_df,
    x_col='path',
    y_col=None,
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

# Manually prepare the dual inputs for model.predict
print("\nPreparing inputs and generating predictions...")

# Get the images from the generator using standard iterator next()
comp_test_generator.reset()
all_images = []
for i in range(math.ceil(comp_test_generator.samples / BATCH_SIZE)):
    batch = next(comp_test_generator)
    all_images.append(batch)
x_images = np.vstack(all_images)

# Get the gender input
x_gender = comp_test_df['male'].values.astype('float32')

Original Competition Test DataFrame:


,id,male
0,1.png,False
1,2.png,False
2,3.png,True
3,4.png,False
4,5.png,False



Processed Competition Test DataFrame:


,id,male,path
0,1.png,False,eliva_data/test/1.png
1,2.png,False,eliva_data/test/2.png
2,3.png,True,eliva_data/test/3.png
3,4.png,False,eliva_data/test/4.png
4,5.png,False,eliva_data/test/5.png


Found 33 validated image filenames.

Preparing inputs and generating predictions...


Finally, we can submit directly to the Kaggle competition using the Kaggle CLI.

In [16]:
#!kaggle competitions submit -c eliva-25-medical -f eliva_submission.csv -m "Final 33-row submission"

### Attempt 2: Improved Architecture (DenseNet121) & Augmentation
Let's try switching the base model to `DenseNet121` and adding slightly more aggressive data augmentation to prevent overfitting and improve generalization.

In [17]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Enhanced Data Augmentation
train_datagen_enhanced = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,          # Increased from 10
    width_shift_range=0.15,     # Increased from 0.1
    height_shift_range=0.15,    # Increased from 0.1
    shear_range=0.15,           # Increased from 0.1
    zoom_range=0.15,            # Increased from 0.1
    brightness_range=[0.7, 1.3],# Wider brightness range
    fill_mode='nearest',
    validation_split=0.2
)

# Recreate generators with enhanced augmentation
train_generator_enhanced = train_datagen_enhanced.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col=['boneage', 'male'],
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    subset='training',
    seed=42
)

validation_generator_enhanced = train_datagen_enhanced.flow_from_dataframe(
    dataframe=train_df,
    x_col='path',
    y_col=['boneage', 'male'],
    target_size=(IMG_WIDTH, IMG_HEIGHT),
    batch_size=BATCH_SIZE,
    class_mode='raw',
    subset='validation',
    seed=42
)

# Wrapper generator to split the combined targets into (gender input, boneage target)
def combined_generator(img_gen):
    for img_batch, targets_batch in img_gen:
        # Explicitly cast to float32 to fix the 'object' dtype error
        boneage_batch = targets_batch[:, 0].astype('float32')
        gender_batch = targets_batch[:, 1].astype('float32')
        yield ((img_batch, gender_batch), boneage_batch)

train_data_generator_dn = combined_generator(train_generator_enhanced)
validation_data_generator_dn = combined_generator(validation_generator_enhanced)

print("Enhanced generators ready.")

Found 10089 validated image filenames.
Found 2522 validated image filenames.
Enhanced generators ready.


In [18]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, concatenate, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.optimizers import Adam

def build_densenet_boneage_model(input_shape=(IMG_WIDTH, IMG_HEIGHT, 3), gender_input_shape=(1,)):
    # Use DenseNet121 as the new base model
    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)
    
    for layer in base_model.layers:
        layer.trainable = False
        
    image_input = Input(shape=input_shape, name='image_input')
    x = base_model(image_input)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    
    gender_input = Input(shape=gender_input_shape, name='gender_input')
    combined_input = concatenate([x, gender_input])
    
    z = Dense(256, activation='relu')(combined_input)
    z = BatchNormalization()(z)
    z = Dropout(0.4)(z)
    z = Dense(128, activation='relu')(z)
    z = BatchNormalization()(z)
    z = Dropout(0.3)(z)
    
    output = Dense(1, activation='linear', name='boneage_output')(z)
    
    model = Model(inputs=[image_input, gender_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_absolute_error', metrics=['mean_absolute_error'])
    
    return model

densenet_model = build_densenet_boneage_model()
densenet_model.summary()

I0000 00:00:1782892021.325700      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 384, 384,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ densenet121         │ (None, 12, 12,    │  7,037,504 │ image_input[0][0] │
│ (Functional)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1024)      │          0 │ densenet121[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 1024)      │      4,096 │ global_average_p… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gender_input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1025)      │          0 │ batch_normalizat… │
│ (Concatenate)       │                   │            │ gender_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    262,656 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     32,896 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ boneage_output      │ (None, 1)         │        129 │ dropout_1[0][0]   │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,338,817 (28.00 MB)

 Trainable params: 298,497 (1.14 MB)

 Non-trainable params: 7,040,320 (26.86 MB)

In [19]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Train the DenseNet model
dn_early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
dn_model_checkpoint = ModelCheckpoint('best_densenet_model.keras', save_best_only=True, monitor='val_loss', mode='min', verbose=1)
dn_reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001, verbose=1)

EPOCHS = 40

print("Starting DenseNet model training...")

history_dn = densenet_model.fit(
    train_data_generator_dn,
    steps_per_epoch=train_generator_enhanced.samples // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=validation_data_generator_dn,
    validation_steps=validation_generator_enhanced.samples // BATCH_SIZE,
    callbacks=[dn_early_stopping, dn_model_checkpoint, dn_reduce_lr]
)

print("DenseNet training complete.")

Starting DenseNet model training...
Epoch 1/40


I0000 00:00:1782892044.531534      68 service.cc:152] XLA service 0x7c1234224980 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782892044.531600      68 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1782892049.002846      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


  2/630 ━━━━━━━━━━━━━━━━━━━━ 44s 70ms/step - loss: 124.1225 - mean_absolute_error: 124.1225   

I0000 00:00:1782892065.038321      68 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


630/630 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 117.8732 - mean_absolute_error: 117.8732
Epoch 1: val_loss improved from None to 50.13439, saving model to best_densenet_model.keras

Epoch 1: finished saving model to best_densenet_model.keras
630/630 ━━━━━━━━━━━━━━━━━━━━ 1150s 2s/step - loss: 97.7493 - mean_absolute_error: 97.7493 - val_loss: 50.1344 - val_mean_absolute_error: 50.1344 - learning_rate: 0.0010
Epoch 2/40


2026-07-01 08:06:24.810562: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-01 08:06:25.046818: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


630/630 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 23.3023 - mean_absolute_error: 23.3023
Epoch 2: val_loss improved from 50.13439 to 26.43618, saving model to best_densenet_model.keras

Epoch 2: finished saving model to best_densenet_model.keras
630/630 ━━━━━━━━━━━━━━━━━━━━ 900s 1s/step - loss: 20.8578 - mean_absolute_error: 20.8578 - val_loss: 26.4362 - val_mean_absolute_error: 26.4362 - learning_rate: 0.0010
Epoch 3/40
630/630 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 18.4239 - mean_absolute_error: 18.4239
Epoch 3: val_loss improved from 26.43618 to 24.24522, saving model to best_densenet_model.keras

Epoch 3: finished saving model to best_densenet_model.keras
630/630 ━━━━━━━━━━━━━━━━━━━━ 885s 1s/step - loss: 18.1304 - mean_absolute_error: 18.1304 - val_loss: 24.2452 - val_mean_absolute_error: 24.2452 - learning_rate: 0.0010
Epoch 4/40
630/630 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 17.5700 - mean_absolute_error: 17.5700

2026-07-01 08:48:13.882810: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-01 08:48:14.120980: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.



Epoch 4: val_loss improved from 24.24522 to 21.53607, saving model to best_densenet_model.keras

Epoch 4: finished saving model to best_densenet_model.keras
630/630 ━━━━━━━━━━━━━━━━━━━━ 904s 1s/step - loss: 17.3208 - mean_absolute_error: 17.3208 - val_loss: 21.5361 - val_mean_absolute_error: 21.5361 - learning_rate: 0.0010
Epoch 5/40
630/630 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 16.9698 - mean_absolute_error: 16.9698
Epoch 5: val_loss improved from 21.53607 to 20.51382, saving model to best_densenet_model.keras

Epoch 5: finished saving model to best_densenet_model.keras
630/630 ━━━━━━━━━━━━━━━━━━━━ 884s 1s/step - loss: 16.9063 - mean_absolute_error: 16.9063 - val_loss: 20.5138 - val_mean_absolute_error: 20.5138 - learning_rate: 0.0010
Epoch 6/40
630/630 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 16.5253 - mean_absolute_error: 16.5253
Epoch 6: val_loss did not improve from 20.51382
630/630 ━━━━━━━━━━━━━━━━━━━━ 851s 1s/step - loss: 16.4168 - mean_absolute_error: 16.4168 - val_loss: 21.6

In [20]:
from tensorflow.keras.models import load_model

# Generate predictions for the new model
print("Loading the best trained DenseNet model...")
best_dn_model = load_model('best_densenet_model.keras')

# Generate predictions (using the x_images and x_gender prepared earlier)
dn_comp_predictions = best_dn_model.predict([x_images, x_gender])

# Create the final submission DataFrame
dn_comp_submission_df = pd.DataFrame({
    'id': comp_test_df['id'],
    'boneage': dn_comp_predictions.flatten()
})

dn_comp_submission_file = 'eliva_densenet_submission.csv'
dn_comp_submission_df.to_csv(dn_comp_submission_file, index=False)
print(f"\nSaved new submission to {dn_comp_submission_file}")
display(dn_comp_submission_df.head())

Loading the best trained DenseNet model...
2/2 ━━━━━━━━━━━━━━━━━━━━ 33s 14s/step

Saved new submission to eliva_densenet_submission.csv


,id,boneage
0,1.png,117.170639
1,2.png,137.486328
2,3.png,133.743195
3,4.png,130.198059
4,5.png,145.809753


In [21]:
!kaggle competitions submit -c eliva-25-medical -f eliva_densenet_submission.csv -m "DenseNet121 architecture submission"

100%|████████████████████████████████████████████| 558/558 [00:00<00:00, 784B/s]
Successfully submitted to ELIVA25-Medical